In [2]:
import dhg

hg = dhg.Hypergraph(5, [(0, 1, 2), (1, 3, 2), (1, 2), (0, 3, 4)]) # type: ignore

g_clique = dhg.Graph.from_hypergraph_clique(hg)
print(g_clique)
print(g_clique.e)

Graph(num_v=5, num_e=8)
([(0, 4), (0, 3), (0, 2), (0, 1), (1, 3), (1, 2), (2, 3), (3, 4)], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0])


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import dhg
from dhg.data import Cora
from dhg.nn import GCNConv

# === 1. 数据准备 ===
data = Cora()
X, y = data['features'], data['labels']
train_mask, test_mask = data['train_mask'], data['test_mask']
num_v, num_classes = data['num_vertices'], data['num_classes']

# 构建超图：1-Hop 邻居法
base_g = dhg.Graph(num_v, data['edge_list'])
hg = dhg.Hypergraph.from_graph_kHop(base_g, k=1)

# === 2. 执行团扩展 (Clique Expansion) ===
# 将超图转为普通图，不增加节点，只增加边
g_clique = dhg.Graph.from_hypergraph_clique(hg)

# === 3. 定义模型 ===
class CliqueGCN(nn.Module):
    def __init__(self, in_ch, hid_ch, out_ch):
        super().__init__()
        self.conv1 = GCNConv(in_ch, hid_ch)
        self.conv2 = GCNConv(hid_ch, out_ch)

    def forward(self, x, g):
        # 团扩展后的节点数与原始 X 完全对应，直接卷积即可
        x = F.relu(self.conv1(x, g))
        x = self.conv2(x, g)
        return x

# === 4. 训练配置 ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CliqueGCN(X.shape[1], 16, num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

X, y = X.to(device), y.to(device)
g_clique = g_clique.to(device)

# === 5. 训练循环 ===
print(f"{'Epoch':<8} | {'Loss':<10} | {'Train Acc':<10} | {'Test Acc':<10}")
print("-" * 50)

for epoch in range(201):
    model.train()
    optimizer.zero_grad()
    
    out = model(X, g_clique)
    loss = F.cross_entropy(out[train_mask], y[train_mask])
    
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        model.eval()
        with torch.no_grad():
            pred = out.argmax(dim=1)
            train_acc = (pred[train_mask] == y[train_mask]).float().mean()
            test_acc = (pred[test_mask] == y[test_mask]).float().mean()
            print(f"{epoch:<8} | {loss.item():<10.4f} | {train_acc:<10.2%} | {test_acc:<10.2%}")

Epoch    | Loss       | Train Acc  | Test Acc  
--------------------------------------------------
0        | 1.9459     | 14.29%     | 13.00%    
20       | 1.9210     | 18.57%     | 14.30%    
40       | 1.8654     | 23.57%     | 17.00%    
60       | 1.7834     | 27.86%     | 19.70%    
80       | 1.7982     | 21.43%     | 18.70%    
100      | 1.7491     | 23.57%     | 18.50%    
120      | 1.6782     | 27.86%     | 17.90%    
140      | 1.7155     | 26.43%     | 20.10%    
160      | 1.7326     | 22.14%     | 20.20%    
180      | 1.6732     | 27.14%     | 20.70%    
200      | 1.6805     | 24.29%     | 19.10%    
